# Chapter 08 — Polars and Apache Arrow for Efficient Data Processing

Polars offers an expression-based DataFrame API and lazy query optimization. Apache Arrow is a columnar memory format for efficient data interchange.

## Prerequisites
Python, Pandas, SQL-style filtering/grouping; install `polars` and `pyarrow` for full examples.

## Learning goals
- Explain eager versus lazy execution and query plans.
- Use Polars expressions to select, filter, add, and aggregate columns.
- Read files lazily and collect results deliberately.
- Understand Arrow tables and interoperability.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Polars vs Pandas vs Arrow
2. Eager DataFrames and expression API
3. Lazy execution
4. Nulls, NaNs, and conversions

### Polars vs Pandas vs Arrow
Pandas is a general-purpose DataFrame library. Polars emphasizes columnar execution, expressions, and parallel query processing. Apache Arrow is a language-independent columnar memory format and ecosystem, not a DataFrame API replacement. Polars is not a drop-in replacement for every Pandas workflow; choose based on workload and integrations.

#### Demonstration 1 — Create a Polars frame (optional)

Run the code and inspect both the output and the assumptions behind it.

In [2]:
try:
 import polars as pl
 frame=pl.DataFrame({"region":["North","South","North"],"revenue":[120,90,80]})
 display(frame.with_columns((pl.col("revenue")*1.1).alias("with_tax")))
except ImportError: print("Install with: pip install polars")

Install with: pip install polars


### Eager DataFrames and expression API
`pl.DataFrame` constructs an eager frame. `select` returns selected/computed columns; `with_columns` adds or replaces; `filter` keeps rows; `group_by().agg()` aggregates; `sort` orders. Expressions: `pl.col`, `pl.lit`, `.cast`, `.alias`, `.sum`, `.mean`, `.is_null`, `.fill_null`, string expressions. Expressions describe work performed by a DataFrame operation.

#### Demonstration 2 — Read CSV in eager mode

Run the code and inspect both the output and the assumptions behind it.

In [3]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
try:
 import polars as pl
 df=pl.read_csv(DATA/"sales.csv"); print(df.shape); display(df.head(3))
except ImportError: print("Install with: pip install polars")

Data directory: /mnt/data/python_data_analysis_book/data
Install with: pip install polars


### Lazy execution
`scan_csv(path)` creates a lazy source; `filter` and `select` build a plan. `collect()` executes/materializes; `explain()` displays the plan. Projection and predicate pushdown may avoid loading unnecessary columns/rows. Lazy mode is useful for multi-step pipelines; eager mode is convenient for small data. Benchmark on representative data.

#### Demonstration 3 — Build an expression pipeline

Run the code and inspect both the output and the assumptions behind it.

In [4]:
try:
 result=(df.with_columns((pl.col("revenue")-pl.col("cost")).alias("recomputed_profit")).filter(pl.col("revenue")>10000).select("order_id","region","revenue","recomputed_profit"))
 display(result.head())
except NameError: print("Run the Polars import cell first")

Run the Polars import cell first


### Nulls, NaNs, and conversions
Polars distinguishes null from floating-point NaN. Use `.is_null`/`.fill_null` for nulls and `.is_nan`/`.fill_nan` for NaN. Convert with `.to_pandas()` or `.to_arrow()` when required; conversion may add memory/time cost. PyArrow is required for some Arrow interoperability paths.

#### Demonstration 4 — Build a lazy query

Run the code and inspect both the output and the assumptions behind it.

In [5]:
try:
 lazy=pl.scan_csv(DATA/"sales.csv").filter(pl.col("region")=="North").group_by("product").agg(pl.col("revenue").sum().alias("total_revenue")).sort("total_revenue",descending=True)
 print(lazy.explain()); display(lazy.collect())
except NameError: print("Install Polars")

Install Polars


#### Additional demonstration 5 — Convert to Arrow

In [6]:
try:
 arrow_table=df.to_arrow(); print(type(arrow_table).__name__,arrow_table.num_rows,arrow_table.num_columns); print(arrow_table.schema)
except Exception as exc: print("Install Polars and PyArrow:",exc)

Install Polars and PyArrow: name 'df' is not defined


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Polars vs Pandas vs Arrow.
- Explain one important concept from: Eager DataFrames and expression API.
- Explain one important concept from: Lazy execution.
- Explain one important concept from: Nulls, NaNs, and conversions.